# RNN 實際應用案例教程

本教程將展示如何將現代RNN應用到實際問題中，包括：
1. 文本生成 (Text Generation)
2. 情感分析 (Sentiment Analysis)
3. 命名實體識別 (Named Entity Recognition)
4. 簡單對話系統 (Simple Chatbot)

每個應用都會包含完整的代碼實現、訓練流程和AI輔助學習指南。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import random

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")

## 應用1: 文本生成 (Character-Level Text Generation)

### 任務描述
給定一個文本前綴，模型自動生成後續文本。這是RNN最經典的應用之一。

### 應用場景
- 詩歌生成
- 代碼自動補全
- 郵件自動回覆
- 創意寫作輔助

In [ ]:
class CharRNN(nn.Module):
    """字符級RNN文本生成模型"""
    
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2):
        super(CharRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 詞嵌入層
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM層
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           batch_first=True, dropout=0.3)
        
        # 輸出層
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        
        if hidden is None:
            output, hidden = self.lstm(embedded)
        else:
            output, hidden = self.lstm(embedded, hidden)
        
        # output: (batch_size, seq_len, hidden_dim)
        output = self.fc(output)  # (batch_size, seq_len, vocab_size)
        
        return output, hidden
    
    def generate(self, start_text, char_to_idx, idx_to_char, max_length=100, temperature=1.0):
        """生成文本
        
        Args:
            start_text: 起始文本
            char_to_idx: 字符到索引的映射
            idx_to_char: 索引到字符的映射
            max_length: 最大生成長度
            temperature: 溫度參數，控制隨機性(越高越隨機)
        """
        self.eval()
        
        with torch.no_grad():
            # 初始化
            chars = [ch for ch in start_text]
            input_seq = torch.tensor([char_to_idx.get(ch, 0) for ch in chars], 
                                    dtype=torch.long).unsqueeze(0).to(device)
            hidden = None
            
            # 生成
            for _ in range(max_length):
                output, hidden = self(input_seq, hidden)
                
                # 取最後一個時間步的輸出
                last_output = output[0, -1, :] / temperature
                probs = F.softmax(last_output, dim=0)
                
                # 採樣下一個字符
                next_char_idx = torch.multinomial(probs, 1).item()
                next_char = idx_to_char[next_char_idx]
                
                chars.append(next_char)
                
                # 準備下一個輸入
                input_seq = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
            
            return ''.join(chars)

print("文本生成模型定義完成！")

### 文本生成示例

In [ ]:
# 示例：訓練一個簡單的文本生成模型
sample_text = """To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them."""

# 構建詞表
chars = sorted(list(set(sample_text)))
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}
vocab_size = len(chars)

print(f"詞表大小: {vocab_size}")
print(f"文本長度: {len(sample_text)}")
print(f"\n示例字符: {chars[:20]}...")

# 創建模型
text_gen_model = CharRNN(vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2)
text_gen_model = text_gen_model.to(device)

# 生成示例（未訓練的模型）
generated = text_gen_model.generate("To be", char_to_idx, idx_to_char, max_length=50, temperature=0.8)
print(f"\n生成文本（未訓練）:\n{generated}")
print("\n💡 提示: 訓練後的模型會生成更有意義的文本")

## 應用2: 情感分析 (Sentiment Analysis)

### 任務描述
判斷一段文本的情感傾向（正面/負面）。

### 應用場景
- 產品評論分析
- 社交媒體監控
- 客戶滿意度調查
- 品牌聲譽管理

In [ ]:
class SentimentRNN(nn.Module):
    """情感分析RNN模型"""
    
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, 
                 num_layers=2, num_classes=2, dropout=0.5):
        super(SentimentRNN, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 使用雙向LSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=True)
        
        self.dropout = nn.Dropout(dropout)
        
        # 雙向LSTM輸出維度是hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)
        
        # output: (batch_size, seq_len, hidden_dim * 2)
        output, (hidden, cell) = self.lstm(embedded)
        
        # 取最後一個時間步的輸出
        last_output = output[:, -1, :]
        
        # 應用dropout
        last_output = self.dropout(last_output)
        
        # 分類
        logits = self.fc(last_output)
        
        return logits

print("情感分析模型定義完成！")

### 情感分析示例

In [ ]:
# 示例數據
sample_reviews = [
    ("This movie is amazing! I loved it.", 1),  # 正面
    ("Terrible film, waste of time.", 0),       # 負面
    ("Great acting and wonderful story.", 1),   # 正面
    ("Boring and disappointing.", 0),           # 負面
]

# 簡單的詞表構建
all_words = []
for review, _ in sample_reviews:
    all_words.extend(review.lower().split())

word_counts = Counter(all_words)
vocab_words = ['<PAD>', '<UNK>'] + [word for word, _ in word_counts.most_common()]
word_to_idx = {word: idx for idx, word in enumerate(vocab_words)}

# 創建模型
sentiment_model = SentimentRNN(len(vocab_words), embedding_dim=50, hidden_dim=64)
sentiment_model = sentiment_model.to(device)

print(f"情感分析詞表大小: {len(vocab_words)}")
print(f"示例詞彙: {vocab_words[:15]}")

# 測試前向傳播
test_input = torch.randint(0, len(vocab_words), (2, 10)).to(device)  # batch=2, seq_len=10
with torch.no_grad():
    output = sentiment_model(test_input)
    print(f"\n模型輸出形狀: {output.shape}")
    print(f"預測概率: {F.softmax(output, dim=1)}")

## 應用3: 命名實體識別 (Named Entity Recognition)

### 任務描述
從文本中識別並分類命名實體（人名、地名、組織名等）。

### 應用場景
- 信息抽取
- 知識圖譜構建
- 智能搜索
- 文檔理解

In [ ]:
class NERModel(nn.Module):
    """命名實體識別模型"""
    
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128,
                 num_tags=5, num_layers=2, dropout=0.3):
        """
        Args:
            num_tags: 標籤數量 (例如: O, B-PER, I-PER, B-LOC, I-LOC)
        """
        super(NERModel, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 雙向LSTM更適合序列標註任務
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=True)
        
        self.dropout = nn.Dropout(dropout)
        
        # 為每個時間步預測標籤
        self.hidden2tag = nn.Linear(hidden_dim * 2, num_tags)
        
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)
        
        # output: (batch_size, seq_len, hidden_dim * 2)
        output, _ = self.lstm(embedded)
        
        output = self.dropout(output)
        
        # 為每個時間步預測標籤
        # tag_space: (batch_size, seq_len, num_tags)
        tag_space = self.hidden2tag(output)
        
        return tag_space

print("NER模型定義完成！")

### NER 示例

In [ ]:
# NER 標籤示例
# O: 非實體
# B-PER: 人名開始
# I-PER: 人名內部
# B-LOC: 地名開始
# I-LOC: 地名內部

tag_to_idx = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-LOC': 3, 'I-LOC': 4}
idx_to_tag = {v: k for k, v in tag_to_idx.items()}

# 示例句子和標註
example_sentence = "John lives in New York"
# John: B-PER
# lives: O
# in: O
# New: B-LOC
# York: I-LOC

# 創建模型
ner_vocab_size = 1000  # 假設詞表大小
ner_model = NERModel(ner_vocab_size, embedding_dim=50, hidden_dim=64, num_tags=len(tag_to_idx))
ner_model = ner_model.to(device)

# 測試
test_input = torch.randint(0, ner_vocab_size, (1, 5)).to(device)  # batch=1, seq_len=5
with torch.no_grad():
    tag_scores = ner_model(test_input)
    predictions = torch.argmax(tag_scores, dim=2)
    
    print(f"輸入形狀: {test_input.shape}")
    print(f"標籤分數形狀: {tag_scores.shape}")
    print(f"預測標籤: {[idx_to_tag[idx.item()] for idx in predictions[0]]}")

## 應用4: 簡單對話系統 (Simple Chatbot)

### 任務描述
基於Seq2Seq架構的簡單問答系統。

### 應用場景
- 客服機器人
- 虛擬助手
- 教育輔導
- 娛樂聊天

In [ ]:
class SimpleChatbot(nn.Module):
    """簡單的對話機器人（Seq2Seq架構）"""
    
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256):
        super(SimpleChatbot, self).__init__()
        
        # 編碼器
        self.encoder_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder_lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        
        # 解碼器
        self.decoder_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.decoder_lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        
        # 輸出層
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def encode(self, x):
        """編碼輸入序列"""
        embedded = self.encoder_embedding(x)
        _, (hidden, cell) = self.encoder_lstm(embedded)
        return hidden, cell
    
    def decode(self, x, hidden, cell):
        """解碼生成回覆"""
        embedded = self.decoder_embedding(x)
        output, (hidden, cell) = self.decoder_lstm(embedded, (hidden, cell))
        prediction = self.fc(output)
        return prediction, hidden, cell
    
    def forward(self, input_seq, target_seq):
        """訓練時的前向傳播"""
        # 編碼
        hidden, cell = self.encode(input_seq)
        
        # 解碼
        predictions, _, _ = self.decode(target_seq, hidden, cell)
        
        return predictions

print("對話系統模型定義完成！")

### 對話系統示例

In [ ]:
# 示例對話數據
qa_pairs = [
    ("Hello", "Hi there!"),
    ("How are you?", "I'm doing well, thanks!"),
    ("What's your name?", "I'm a chatbot."),
    ("Goodbye", "See you later!"),
]

# 創建模型
chatbot_vocab_size = 500  # 假設詞表大小
chatbot = SimpleChatbot(chatbot_vocab_size, embedding_dim=64, hidden_dim=128)
chatbot = chatbot.to(device)

# 測試前向傳播
test_input = torch.randint(0, chatbot_vocab_size, (2, 5)).to(device)  # batch=2, seq_len=5
test_target = torch.randint(0, chatbot_vocab_size, (2, 6)).to(device)

with torch.no_grad():
    output = chatbot(test_input, test_target)
    print(f"輸入形狀: {test_input.shape}")
    print(f"目標形狀: {test_target.shape}")
    print(f"輸出形狀: {output.shape}")
    print(f"\n💡 輸出形狀解釋: (batch_size, target_seq_len, vocab_size)")

## 🤖 AI 輔助學習指南

### 💡 應用選擇指南

| 任務類型 | 推薦架構 | 關鍵點 | 難度 |
|---------|---------|-------|------|
| 文本生成 | LSTM/GRU | 需要創造力，溫度參數很重要 | ⭐⭐⭐ |
| 情感分析 | BiLSTM | 雙向上下文很重要 | ⭐⭐ |
| NER | BiLSTM+CRF | 序列標註，需要全局信息 | ⭐⭐⭐⭐ |
| 對話系統 | Seq2Seq | 編碼-解碼架構 | ⭐⭐⭐⭐⭐ |

### 🔍 深入理解

#### 1. 文本生成中的溫度(Temperature)

溫度控制生成文本的隨機性：

```python
# 低溫度 (0.5): 更確定性，重複性高
logits / 0.5  # 增大logits差異

# 高溫度 (1.5): 更隨機，創造性高
logits / 1.5  # 減小logits差異
```

**實際應用：**
- 代碼補全: temperature=0.3-0.5 (要準確)
- 詩歌生成: temperature=0.8-1.2 (要創意)
- 對話系統: temperature=0.6-0.8 (平衡)

#### 2. 情感分析的雙向性

為什麼使用BiLSTM？

```
句子: "The movie is not bad"
前向: The → movie → is → not → bad
後向: bad → not → is → movie → The
```

只看前向可能在"not"處判斷為負面，但後向知道"bad"被"not"否定了。

#### 3. NER的BIO標註

```
句子: "Apple CEO Tim Cook visited New York"
標註: B-ORG O  B-PER I-PER O      B-LOC I-LOC
```

- **B**(Begin): 實體開始
- **I**(Inside): 實體內部
- **O**(Outside): 非實體

### 🛠️ 實戰技巧

#### 處理不平衡數據

```python
# 情感分析中正負樣本不平衡
class_weights = torch.tensor([1.0, 3.0])  # 負樣本權重更高
criterion = nn.CrossEntropyLoss(weight=class_weights)
```

#### 防止過擬合

```python
# 1. Dropout
self.dropout = nn.Dropout(0.5)

# 2. 早停
if val_loss > best_val_loss:
    patience_counter += 1
    if patience_counter >= patience:
        break

# 3. 數據增強
# 同義詞替換、隨機刪除等
```

#### 加速訓練

```python
# 1. 使用pack_padded_sequence處理變長序列
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 2. 梯度累積
for i, batch in enumerate(dataloader):
    loss = model(batch)
    loss = loss / accumulation_steps
    loss.backward()
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

### 📊 性能評估

#### 文本生成
- **困惑度(Perplexity)**: 越低越好
- **人工評估**: 流暢性、連貫性、創造性

#### 情感分析
- **準確率(Accuracy)**
- **F1分數**: 特別是類別不平衡時
- **ROC-AUC**

#### NER
- **實體級F1**: 只有完全正確的實體才算對
- **詞元級準確率**: 每個詞元的標註準確率

#### 對話系統
- **BLEU分數**: 與參考回覆的相似度
- **人工評估**: 相關性、流暢性、信息量

### 🎯 常見問題

**Q: 生成的文本一直重複怎麼辦？**

A: 
1. 增加溫度參數
2. 添加重複懲罰
3. 使用nucleus sampling (top-p)
4. 訓練更長時間or更多數據

**Q: 情感分析準確率不高？**

A:
1. 嘗試預訓練詞嵌入(Word2Vec, GloVe)
2. 增加訓練數據
3. 使用更深的網絡
4. 添加注意力機制

**Q: NER標註不準確？**

A:
1. 添加CRF層考慮標籤轉移
2. 使用字符級特徵
3. 預訓練BERT等模型
4. 增加標註數據質量

## 🎓 實戰項目建議

### 初級項目
1. **詩歌生成器**: 在唐詩三百首上訓練，生成古詩
2. **電影評論分類**: 使用IMDB數據集
3. **簡單問答機器人**: FAQ系統

### 中級項目
1. **新聞摘要生成**: Seq2Seq with Attention
2. **多類情感分析**: 正面/中性/負面/非常正面/非常負面
3. **中文NER**: 識別人名、地名、機構名

### 高級項目
1. **多輪對話系統**: 帶上下文記憶
2. **機器翻譯**: 中英互譯
3. **代碼生成**: 根據註釋生成代碼

## 小結

本教程介紹了RNN在四個實際應用中的使用：

1. **文本生成**: 創造性任務，關注生成質量和多樣性
2. **情感分析**: 分類任務，關注上下文理解
3. **NER**: 序列標註，關注局部和全局信息
4. **對話系統**: Seq2Seq任務，關注編碼解碼架構

每個應用都有其特點和挑戰，需要針對性地調整模型架構和訓練策略。

## 練習

1. 實現一個完整的文本生成pipeline，包括數據處理、訓練、生成
2. 在真實數據集(如IMDb)上訓練情感分析模型
3. 為NER模型添加CRF層，提升性能
4. 實現帶注意力機制的Seq2Seq對話系統
5. 比較不同溫度參數對文本生成的影響

## 資源推薦

**數據集：**
- 情感分析: [IMDb](http://ai.stanford.edu/~amaas/data/sentiment/), [Yelp Reviews](https://www.yelp.com/dataset)
- NER: [CoNLL-2003](https://www.clips.uantwerpen.be/conll2003/ner/), [OntoNotes](https://catalog.ldc.upenn.edu/LDC2013T19)
- 對話: [Cornell Movie Dialogs](https://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html)

**工具：**
- [Hugging Face Transformers](https://huggingface.co/transformers/): 預訓練模型
- [spaCy](https://spacy.io/): NLP工具包
- [NLTK](https://www.nltk.org/): 文本處理